# Healthcare Cost Predictor — Model Training

Trains a regression model on `insurance.csv` (age, sex, bmi, children, smoker, region → charges), compares a few model types, and saves the best one for reuse (e.g. in a Streamlit app).

**Run the cells top to bottom.** The first code cell lets you upload `insurance.csv` directly in Colab.

## 1. Install dependencies

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib joblib


## 2. Upload the dataset
Run this cell and select `insurance.csv` from your computer when prompted.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select insurance.csv in the dialog


## 3. Load and inspect the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

df = pd.read_csv("insurance.csv")
print(df.info())
df.head()


## 4. Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

NUMERIC_FEATURES = ["age", "bmi", "children"]
CATEGORICAL_FEATURES = ["sex", "smoker", "region"]
TARGET = "charges"

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 5. Build the preprocessing pipeline
Numeric features are scaled; categorical features are one-hot encoded. Bundling this into a `ColumnTransformer` means the exact same preprocessing is applied automatically at prediction time on new data.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(drop="first"), CATEGORICAL_FEATURES),
])


## 6. Train and compare candidate models
Charges are driven by a few nonlinear interactions (smoker status especially), so tree-based models tend to do well. Linear regression is included as an interpretable baseline.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

candidates = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42
    ),
}

results = {}
fitted_pipelines = {}

for name, model in candidates.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2}
    fitted_pipelines[name] = pipe

results_df = pd.DataFrame(results).T.sort_values("RMSE")
results_df.round(2)


## 7. Pick the best model and save it

In [ ]:
best_name = results_df.index[0]
best_pipeline = fitted_pipelines[best_name]
print(f"Best model: {best_name}")
print(results_df.loc[best_name].round(2).to_dict())

joblib.dump(best_pipeline, "healthcare_cost_model.joblib")
print("Saved trained pipeline to healthcare_cost_model.joblib")


## 8. Predicted vs actual plot

In [ ]:
best_preds = best_pipeline.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_preds, alpha=0.6)
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
plt.plot(lims, lims, "r--", label="perfect prediction")
plt.xlabel("Actual charges ($)")
plt.ylabel("Predicted charges ($)")
plt.title(f"{best_name}: Predicted vs Actual Charges (R^2={results[best_name]['R2']:.3f})")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Feature importance (if supported by the best model)

In [ ]:
model_step = best_pipeline.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    feature_names = (
        NUMERIC_FEATURES
        + list(best_pipeline.named_steps["preprocess"]
               .named_transformers_["cat"]
               .get_feature_names_out(CATEGORICAL_FEATURES))
    )
    importances = pd.Series(model_step.feature_importances_, index=feature_names)
    importances = importances.sort_values(ascending=False)
    print(importances.round(3))

    plt.figure(figsize=(8, 5))
    importances.plot(kind="barh")
    plt.gca().invert_yaxis()
    plt.title(f"Feature Importance ({best_name})")
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_name} does not expose feature_importances_.")


## 10. Predict on a new example

In [ ]:
new_person = pd.DataFrame([{
    "age": 35,
    "bmi": 28.0,
    "children": 2,
    "sex": "male",
    "smoker": "no",
    "region": "southeast",
}])

predicted_cost = best_pipeline.predict(new_person)[0]
print(f"Predicted healthcare cost for example person: ${predicted_cost:,.2f}")


## 11. Download the trained model
Run this to download `healthcare_cost_model.joblib` to your computer — drop it next to `app.py` to use it in the Streamlit app.

In [ ]:
from google.colab import files

files.download("healthcare_cost_model.joblib")


## 12. Write out `app.py` and `requirements.txt`

This generates the Streamlit app files for you. **Note:** Streamlit apps don't run inside notebook cells — this cell just writes the files to disk so you can download them and run them locally (or deploy them) with `streamlit run app.py`.

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Healthcare Cost Predictor", page_icon="🏥", layout="centered")

# -----------------------------------------------------------------
# Load the trained pipeline (preprocessing + model bundled together)
# -----------------------------------------------------------------
@st.cache_resource
def load_model():
    return joblib.load("healthcare_cost_model.joblib")

model = load_model()

st.title("🏥 Healthcare Cost Predictor")
st.write(
    "Estimate annual insurance charges based on personal and lifestyle "
    "factors, using a Gradient Boosting model trained on historical data."
)

st.divider()

# -----------------------------------------------------------------
# Input form
# -----------------------------------------------------------------
col1, col2 = st.columns(2)

with col1:
    age = st.number_input("Age", min_value=18, max_value=100, value=35, step=1)
    bmi = st.number_input("BMI", min_value=10.0, max_value=60.0, value=27.5, step=0.1)
    children = st.number_input("Number of children", min_value=0, max_value=10, value=0, step=1)

with col2:
    sex = st.selectbox("Sex", ["male", "female"])
    smoker = st.selectbox("Smoker", ["no", "yes"])
    region = st.selectbox("Region", ["northeast", "northwest", "southeast", "southwest"])

st.divider()

if st.button("Predict Cost", type="primary", use_container_width=True):
    input_df = pd.DataFrame([{
        "age": age,
        "bmi": bmi,
        "children": children,
        "sex": sex,
        "smoker": smoker,
        "region": region,
    }])

    prediction = model.predict(input_df)[0]

    st.metric("Predicted Annual Healthcare Cost", f"${prediction:,.2f}")

    if smoker == "yes":
        st.warning(
            "Smoker status has the largest impact on predicted cost — "
            "it typically accounts for the majority of the difference in charges."
        )

    with st.expander("See input used for this prediction"):
        st.dataframe(input_df, use_container_width=True)

st.divider()
st.caption(
    "Model: Gradient Boosting Regressor · Trained on 1,338 records "
    "(age, sex, BMI, children, smoker status, region → charges)."
)


In [ ]:
%%writefile requirements.txt
streamlit==1.38.0
pandas==2.2.2
scikit-learn==1.5.1
joblib==1.4.2


## 13. Download `app.py` and `requirements.txt`
Run this to download both files to your computer. Place them in the same folder as `healthcare_cost_model.joblib` (downloaded in step 11), then run:

```
pip install -r requirements.txt
streamlit run app.py
```

In [ ]:
from google.colab import files

files.download("app.py")
files.download("requirements.txt")
